In [12]:
import yfinance as yf
from datetime import datetime

# Daten laden mit fixer Struktur
stock = "BTC-USD"
end = datetime.now()
start = datetime(end.year-20, end.month, end.day)

# WICHTIG: multi_level_index=False
stock_data = yf.download(stock, start=start, end=end, multi_level_index=False)

# Jetzt funktioniert das hier wieder:
prices = stock_data['Close'] 
print(prices.head())


[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BTC-USD']: TypeError("'NoneType' object is not subscriptable")


Series([], Name: Close, dtype: float64)


In [13]:
stock_data.head()

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,


In [14]:
stock_data.tail()

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,


In [20]:
import yfinance as yf
import pandas as pd

# Nutze Ticker-Objekt statt direktem Download (oft stabiler)
ticker = yf.Ticker("BTC-USD")
stock_data = ticker.history(period="max") # Holt alle verfügbaren Daten

if stock_data.empty:
    # Falls history() fehlschlägt, probier den klassischen Weg mit Korrektur
    stock_data = yf.download("BTC-USD", start="2015-01-01", multi_level_index=False)

if stock_data is None or stock_data.empty:
    print("❌ Fehler: Yahoo Finance liefert keine Daten. Prüfe deine Internetverbindung oder das yfinance-Update.")
else:
    print(f"✅ Daten erfolgreich geladen: {len(stock_data)} Zeilen.")


TypeError: 'NoneType' object is not subscriptable

In [17]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
import joblib

# Modell und Scaler laden
model = load_model('btc_model.h5')
scaler = joblib.load('scaler.gz')

# Vorbereitung
prediction_days = 60
test_start_date = "2022-12-31"

# Sicherstellen, dass wir nur die Close-Spalte als 2D-Array haben
full_prices = stock_data['Close'].values.reshape(-1, 1)

# Jetzt schlägt der Scaler nicht mehr fehl, weil full_prices nicht leer ist
scaled_full_prices = scaler.transform(full_prices)

# Vorhersagen generieren
predicted_prices = []
# Index finden, ab dem wir plotten wollen (Ende 2022)
start_idx = len(stock_data[stock_data.index < test_start_date])

for i in range(start_idx, len(scaled_full_prices)):
    # Fenster von 60 Tagen extrahieren
    x_input = scaled_full_prices[i-prediction_days:i, 0]
    x_input = np.reshape(x_input, (1, prediction_days, 1))
    
    # Vorhersage (verbose=0 unterdrückt Log-Ausgaben)
    pred = model.predict(x_input, verbose=0)
    predicted_prices.append(scaler.inverse_transform(pred)[0][0])

# Plotten
plt.figure(figsize=(12, 6))
plt.plot(stock_data.index, stock_data['Close'], label="Echter Kurs", color='black', alpha=0.3)
plt.plot(stock_data.index[start_idx:], predicted_prices, label="Modell-Vorhersage", color='red')

plt.title("BTC Prediction Check")
plt.legend()
plt.show()


ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by MinMaxScaler.